# G-8 | Member 2 - CNN for Alzheimer's MRI Classification
**Is ek notebook me sab hai:** data prep, CNN model, training, graphs, confusion matrix, architecture diagram aur report section.

**Colab me chalane ke steps:**
1. Runtime > Change runtime type > **T4 GPU**
2. Left side Files panel me `archive.zip` upload karo
3. Cells ko upar se neeche tak chalao (Runtime > Run all)

## 0. Setup aur dataset unzip

In [ ]:
!pip install -q scikit-learn pandas matplotlib pillow
import os
if not os.path.exists('/content/data'):
    !unzip -q /content/archive.zip -d /content/data
!ls /content/data/combined_images

## Imports

In [ ]:
import argparse
import hashlib
import json
import os
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from matplotlib.patches import FancyBboxPatch
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
EXTS = {".jpg", ".jpeg", ".png"}

## 1. Data Preparation

In [ ]:
def build_split(data_dir, csv_path, val=0.15, test=0.15):
    """Scan -> duplicate hatao -> stratified split -> CSV (agar CSV hai to wahi reuse hoti hai)."""
    if os.path.exists(csv_path):
        return pd.read_csv(csv_path)
    data_dir = Path(data_dir)
    classes = sorted(d.name for d in data_dir.iterdir() if d.is_dir())
    if not classes:
        raise SystemExit(f"'{data_dir}' me class folders nahi mile.")
    rows, seen, dups = [], set(), 0
    for label, cls in enumerate(classes):
        for f in sorted((data_dir / cls).iterdir()):
            if f.suffix.lower() not in EXTS:
                continue
            h = hashlib.md5(f.read_bytes()).hexdigest()
            if h in seen:
                dups += 1
                continue
            seen.add(h)
            rows.append((str(f), cls, label))
    df = pd.DataFrame(rows, columns=["path", "class_name", "label"])
    tr, tmp = train_test_split(df, test_size=val + test, stratify=df["label"], random_state=SEED)
    va, te = train_test_split(tmp, test_size=test / (val + test), stratify=tmp["label"], random_state=SEED)
    out = pd.concat([tr.assign(split="train"), va.assign(split="val"), te.assign(split="test")]).reset_index(drop=True)
    out.to_csv(csv_path, index=False)
    print(f"Duplicates removed: {dups} | Images kept: {len(out)}")
    return out


class MRIDataset(Dataset):
    def __init__(self, df, tf):
        self.paths, self.labels, self.tf = df["path"].tolist(), df["label"].tolist(), tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")  # grayscale/RGB dono -> 3 channel
        return self.tf(img), self.labels[i]


def make_tf(size, augment):
    ops = [transforms.Resize((size, size))]
    if augment:
        ops.append(transforms.RandomRotation(10))
    ops += [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
    return transforms.Compose(ops)


def get_loaders(df, batch, size, workers):
    pin = torch.cuda.is_available()

    def mk(split, augment, shuffle):
        return DataLoader(MRIDataset(df[df["split"] == split], make_tf(size, augment)),
                          batch_size=batch, shuffle=shuffle, num_workers=workers, pin_memory=pin)

    return mk("train", True, True), mk("val", False, False), mk("test", False, False)

## 2. Cnn Model

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout),
                         nn.ReLU(inplace=True), nn.MaxPool2d(2))


class CNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(conv_block(3, 32), conv_block(32, 64),
                                      conv_block(64, 128), conv_block(128, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.5), nn.Linear(256, 128),
                                        nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(128, num_classes))

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

## 3. Training / Evaluation

In [ ]:
def run_epoch(model, loader, criterion, device, optimizer=None):
    train = optimizer is not None
    model.train(train)
    loss_sum, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return loss_sum / n, correct / n


def predict_all(model, loader, device):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            ps += model(x.to(device)).argmax(1).cpu().tolist()
            ys += y.tolist()
    return np.array(ys), np.array(ps)

## 4. Plots

In [ ]:
def plot_curves(hist, path):
    ep = range(1, len(hist["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(ep, hist["train_loss"], label="Train"); ax[0].plot(ep, hist["val_loss"], label="Validation")
    ax[0].set_title("CNN Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
    ax[1].plot(ep, hist["train_acc"], label="Train"); ax[1].plot(ep, hist["val_acc"], label="Validation")
    ax[1].set_title("CNN Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


def plot_confusion(cm, classes, path):
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=30, ha="right"); ax.set_yticklabels(classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("CNN Confusion Matrix (Test set)")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


def plot_architecture(path, n_params, n_classes):
    blocks = [("Input\nMRI image\n3 x 224 x 224", "#d9d9d9"),
              ("Conv 3x3 (32)\nBN + ReLU\nMaxPool\n32x112x112", "#a8c8f0"),
              ("Conv 3x3 (64)\nBN + ReLU\nMaxPool\n64x56x56", "#a8c8f0"),
              ("Conv 3x3 (128)\nBN + ReLU\nMaxPool\n128x28x28", "#a8c8f0"),
              ("Conv 3x3 (256)\nBN + ReLU\nMaxPool\n256x14x14", "#a8c8f0"),
              ("Global Avg\nPool\n256", "#f5d58a"),
              ("Dropout 0.5\nFC 256->128\nReLU", "#b9e0b0"),
              ("Dropout 0.3\nFC 128->%d" % n_classes, "#b9e0b0"),
              ("Output\n%d classes" % n_classes, "#d9d9d9")]
    fig, ax = plt.subplots(figsize=(17, 3.6))
    ax.set_xlim(0, len(blocks) * 1.9); ax.set_ylim(0, 3); ax.axis("off")
    for i, (t, c) in enumerate(blocks):
        x = i * 1.9 + 0.15
        ax.add_patch(FancyBboxPatch((x, 0.5), 1.6, 2.0, boxstyle="round,pad=0.03", fc=c, ec="black"))
        ax.text(x + 0.8, 1.5, t, ha="center", va="center", fontsize=9)
        if i < len(blocks) - 1:
            ax.annotate("", xy=(x + 1.9, 1.5), xytext=(x + 1.63, 1.5), arrowprops=dict(arrowstyle="->"))
    ax.text(len(blocks) * 0.95, 2.8, f"CNN Architecture ({n_params:,} trainable parameters)",
            ha="center", fontsize=12, fontweight="bold")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()

## Main

In [ ]:
def main(cfg):
    a = cfg

    os.makedirs(a.out_dir, exist_ok=True)
    torch.manual_seed(SEED); np.random.seed(SEED)
    device = torch.device(a.device)

    df = build_split(a.data_dir, os.path.join(a.out_dir, "split.csv"))
    classes = df.sort_values("label")["class_name"].unique().tolist()
    print("\nClass distribution:\n", df.groupby(["class_name", "split"]).size().unstack(), "\n")
    train_dl, val_dl, test_dl = get_loaders(df, a.batch_size, a.img_size, a.workers)

    model = CNN(len(classes)).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Classes: {classes}\nDevice: {device} | Trainable parameters: {n_params:,}\n")
    plot_architecture(os.path.join(a.out_dir, "cnn_architecture.png"), n_params, len(classes))

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=a.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    weights_path = os.path.join(a.out_dir, "cnn_model.pth")
    best_val, epoch_times = 0.0, []
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    t_start = time.time()
    for ep in range(1, a.epochs + 1):
        t0 = time.time()
        tl, ta = run_epoch(model, train_dl, criterion, device, optimizer)
        vl, va = run_epoch(model, val_dl, criterion, device)
        scheduler.step(vl)
        if device.type == "cuda":
            torch.cuda.synchronize()
        epoch_times.append(time.time() - t0)
        for k, v in zip(hist, (tl, vl, ta, va)):
            hist[k].append(v)
        if va > best_val:
            best_val = va
            torch.save({"model_state": model.state_dict(), "classes": classes}, weights_path)
        print(f"Epoch {ep:02d}/{a.epochs} | train loss {tl:.4f} acc {ta:.4f} | "
              f"val loss {vl:.4f} acc {va:.4f} | {epoch_times[-1]:.1f}s")
    total = time.time() - t_start

    # ---- test set par final evaluation (best model) ----
    model.load_state_dict(torch.load(weights_path, map_location=device)["model_state"])
    y, pred = predict_all(model, test_dl, device)
    test_acc = float((y == pred).mean())
    cm = confusion_matrix(y, pred)
    report = classification_report(y, pred, target_names=classes, digits=4)
    peak_mem = torch.cuda.max_memory_allocated() / 1024**2 if device.type == "cuda" else None

    plot_curves(hist, os.path.join(a.out_dir, "cnn_training_curves.png"))
    plot_confusion(cm, classes, os.path.join(a.out_dir, "cnn_confusion_matrix.png"))
    with open(os.path.join(a.out_dir, "cnn_test_report.txt"), "w") as f:
        f.write(f"Test accuracy: {test_acc:.4f}\n\n{report}\nConfusion matrix:\n{cm}\n")
    with open(os.path.join(a.out_dir, f"cnn_results_{device.type}.json"), "w") as f:
        json.dump({"device": device.type, "epochs": a.epochs, "batch_size": a.batch_size,
                   "img_size": a.img_size, "trainable_params": n_params,
                   "total_train_time_s": total, "avg_epoch_time_s": sum(epoch_times) / len(epoch_times),
                   "peak_gpu_memory_mb": peak_mem, "best_val_acc": best_val,
                   "test_acc": test_acc, "history": hist}, f, indent=2)

    print(f"\nTotal training time: {total:.1f}s | Best val acc: {best_val:.4f} | Test acc: {test_acc:.4f}")
    print(report)
    print(f"Saari files '{a.out_dir}/' me save ho gayi.")

## Run karo (CPU vs GPU ke liye `device` badlo)

In [ ]:
import argparse, torch
cfg = argparse.Namespace(
    data_dir='/content/data/combined_images',
    out_dir='outputs_cnn',
    epochs=1,            # pehle 1 se test karo, phir 15 karo
    batch_size=32, lr=1e-3, img_size=224, workers=2,
    device='cuda' if torch.cuda.is_available() else 'cpu',   # 'cpu' ya 'cuda'
)
main(cfg)

## Graphs dekho

In [ ]:
from IPython.display import Image, display
for f in ['cnn_architecture','cnn_training_curves','cnn_confusion_matrix']:
    display(Image(f'outputs_cnn/{f}.png'))

## Results download karo

In [ ]:
!zip -qr outputs_cnn.zip outputs_cnn
from google.colab import files
files.download('outputs_cnn.zip')

---
# CNN Model (Member 2 Report Section)

## 1. Deep Learning and CNNs
Deep learning uses neural networks with many layers that learn features directly from data. A Convolutional Neural Network (CNN) is designed for images: instead of connecting every pixel to every neuron, it slides small learnable filters over the image, so the same filter detects a pattern (an edge, a texture, a region of brain shrinkage) anywhere in the scan.

Main building blocks:
- **Convolution layer:** applies learnable 3x3 filters and produces feature maps. Early layers capture edges and textures; deeper layers capture larger structures.
- **Batch normalisation:** normalises layer outputs, which stabilises and speeds up training.
- **Activation (ReLU):** f(x) = max(0, x). Adds non-linearity so the network can learn complex patterns.
- **Pooling (MaxPool 2x2):** halves the spatial size, reduces computation and gives some tolerance to small shifts.
- **Global average pooling:** turns each feature map into one number, which reduces parameters compared with flattening.
- **Dropout:** randomly switches off neurons during training to reduce overfitting.
- **Fully connected layers:** combine the learned features and output one score per class.

## 2. Architecture used
Input: 3 x 224 x 224 (grayscale MRI converted to 3 channels, resized and normalised by the data pipeline). Output: 4 classes (NonDemented, VeryMildDemented, MildDemented, ModerateDemented).

| Stage | Layers | Output shape | Parameters |
|---|---|---|---|
| Block 1 | Conv(3->32), BN, ReLU, MaxPool | 32 x 112 x 112 | 960 |
| Block 2 | Conv(32->64), BN, ReLU, MaxPool | 64 x 56 x 56 | 18,624 |
| Block 3 | Conv(64->128), BN, ReLU, MaxPool | 128 x 28 x 28 | 74,112 |
| Block 4 | Conv(128->256), BN, ReLU, MaxPool | 256 x 14 x 14 | 295,680 |
| Pooling | Global average pool | 256 | 0 |
| Classifier | Dropout(0.5), FC 256->128, ReLU, Dropout(0.3), FC 128->4 | 4 | 33,412 |
| **Total** | | | **422,788** |

Diagram: `results/graphs/cnn_architecture.png`.

## 3. Training methodology
- **Loss function:** Cross-entropy loss, the standard choice for multi-class classification.
- **Optimizer:** Adam, learning rate 0.001, with ReduceLROnPlateau (learning rate halves if validation loss does not improve for 2 epochs).
- **Batch size:** 32. **Epochs:** 15 (configurable).
- **Data:** the shared train/validation/test split prepared by Member 1 (70/15/15, stratified, duplicates removed, seed 42). Light augmentation (rotation up to 10 degrees) is applied to the training set only.
- **Model selection:** the weights with the best validation accuracy are saved to `models/cnn_model.pth`. The test set is used only once, at the end.
- **Monitoring:** training and validation loss and accuracy are recorded every epoch and plotted in `results/graphs/cnn_training_curves.png`.

## 4. Implementation
Framework: PyTorch. Files: `src/cnn_model.py` (model definition) and `src/train.py` (training loop, saving, plotting, timing). Results, including total training time and per-epoch time, are written to `results/cnn_results_<device>.json`.

## 5. Results (fill in after running)
| Item | Value |
|---|---|
| Device / hardware | |
| Total training time | |
| Best validation accuracy | |
| Test accuracy | |

All values must be copied from `results/cnn_results_*.json`, not estimated.

## 6. Likely viva questions
1. Why use BatchNorm and Dropout? (Stable training; reduce overfitting.)
2. Why global average pooling instead of flattening? (Far fewer parameters, less overfitting.)
3. What does the loss curve tell you if validation loss rises while training loss falls? (Overfitting.)
4. Why is the test set not used while training? (To give an unbiased estimate of performance.)
5. Why 3x3 filters and increasing channels (32 to 256)? (Small filters are efficient; deeper layers need more filters for richer features.)
